# Meeting 1: PokerBench pipeline and exact-candidate baseline

A consolidated version of the team's notebooks: inspect raw training data, build answer-free
inputs, validate every row, split reproducibly, and evaluate a CPU baseline. Reusable logic
lives in `poker_decision`; this notebook calls the same workflow as the command-line runner.

From the repository root:
```sh
uv sync --locked --extra analysis
uv run --extra analysis jupyter lab notebooks/01_pokerbench_pipeline.ipynb
```
Select the project environment, restart the kernel, and **Run All**. No GPU or API key.
The first run downloads only the two training CSVs (about 90 MB). Both are pinned to a
revision and SHA-256 checked, including cached files. The official test set is never loaded.

Sources: [PokerBench](https://huggingface.co/datasets/RZ412/PokerBench),
[publisher schema](https://github.com/pokerllm/pokerbench#dataset-structure).
See `docs/meeting1-integration.md` for the synthesis of all eight unique submissions.

This measures agreement with reference actions, not profitability or opponent adaptation.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

from poker_decision.data import TRAIN_FILES, action_type, build_inputs, read_rows
from poker_decision.pipeline import prepare, run_baselines

SEED = 42
VALIDATION_FRACTION = 0.2
SAMPLE_PER_KIND = 5_000  # None uses all rows; every source row is validated either way.
RUN_BASELINE = True
ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside the Poker-Decision-Model repository.")
PROCESSED = ROOT / "data/processed/pokerbench_meeting1"
ARTIFACTS = ROOT / "artifacts/pokerbench_meeting1"

## 1. Load, inspect, and audit all training rows

`prepare` records raw previews, inferred column types, missing cells, blank strings, and
exact action frequencies. CSV fields remain strings during parsing; types are inferred
for inspection. An empty preflop history is valid. Blank cards or targets are failures.

Every row becomes an example or a failure with its source row ID and error. Failures are
saved under `data/processed/pokerbench_meeting1/*_parse_failures.jsonl`; **any failure stops
this run before splitting or training**. Older successful outputs are not evidence that
a failed new run succeeded.

Only after validating all rows do we select a fixed-seed working sample.

In [ ]:
train, validation, manifest = prepare(
    ROOT,
    sample_per_kind=SAMPLE_PER_KIND,
    seed=SEED,
    validation_fraction=VALIDATION_FRACTION,
)
for kind in TRAIN_FILES:
    inspection = json.loads((PROCESSED / f"{kind}_inspection.json").read_text())
    print(f"\n{kind}: {inspection['row_count']:,} raw rows")
    display(pd.DataFrame(inspection["raw_preview"]))
    display(pd.DataFrame.from_dict(inspection["columns"], orient="index"))
    display(pd.Series(inspection["action_frequencies"], name="count").sort_values(ascending=False))
    print("Full-source duplicate audit:", manifest["sources"][kind]["diagnostics"])

## 2. Decision examples and preprocessing

The model receives exactly `DecisionInputs(context, candidates)`. `target_index`, source ID,
and street metadata are separate. Candidate actions preserve order and exact sizing;
`2.0bb` becomes `raise 2bb`, `Bet 17` becomes `bet 17bb`. Checks, calls, folds, and all-ins
remain distinct. We never add the answer to the menu to repair a mismatch.

Postflop contexts contain only cards already dealt in `postflop_action`. The parser verifies
the visible cards against their board columns and checks the street's board size. Future
turn/river columns and `aggressor_position` are excluded from inputs. IP/OOP stays as
provided, without inventing seat or stack information. Card case is preserved.

Repeated rows and conflicting labels are counted and retained; none are silently removed.

In [ ]:
for example in [train[0], next(e for e in train if e.street != "preflop")]:
    print(json.dumps(example.to_record(), indent=2))

## 3. Validation and answer-leakage checks

The package rejects empty or duplicate normalized candidates, targets that fail to match
exactly one candidate, invalid cards or state numbers, and malformed histories. Unit tests
cover target removal, added answer fields, future-card changes, and malformed records.

The answer can naturally occur as an action word in history or in the candidate list.
The constraint is that the reference label is never copied into an extra input field.

In [ ]:
for example in train + validation:
    assert example.inputs.candidates
    assert len(set(example.inputs.candidates)) == len(example.inputs.candidates)
    assert 0 <= example.target_index < len(example.inputs.candidates)
    assert set(example.inputs.__dataclass_fields__) == {"context", "candidates"}

for kind, filename in TRAIN_FILES.items():
    row = read_rows(ROOT / "data/raw/pokerbench" / filename)[0]
    original = build_inputs(row, kind)
    row.pop("correct_decision")
    row["answer"] = "THIS_MUST_NOT_ENTER_THE_INPUT"
    assert build_inputs(row, kind) == original
print("All selected examples valid; model inputs do not require the label.")

## 4. Reproducible train/validation split

Distinct contexts are sorted then shuffled with seed 42. Every occurrence of an identical
context goes to one side, even if candidate menus or labels differ. The 20% fraction refers
to distinct contexts, so row counts may differ from 80/20. Sampling and membership are saved
in the manifest. No stratification or test-derived weighting is used.

This protects exact contexts only. Related scenarios or suit-equivalent hands can still
cross the split; later research needs a stricter scenario-group evaluation.

In [ ]:
assert {e.inputs.context for e in train}.isdisjoint(e.inputs.context for e in validation)
assert {e.source_id for e in train}.isdisjoint(e.source_id for e in validation)
print(f"Train: {len(train):,}; validation: {len(validation):,}; seed: {SEED}")
print("Official test used:", manifest["official_test_used"])

summary = pd.DataFrame(
    [
        {"split": split, "street": e.street, "action_type": action_type(e.target_action)}
        for split, rows in [("train", train), ("validation", validation)]
        for e in rows
    ]
)
display(pd.crosstab(summary["action_type"], [summary["split"], summary["street"]]))

# Audit how much the supplied menu alone reveals about the answer.
menus = pd.DataFrame(
    [
        {
            "menu_types": str(sorted(action_type(c) for c in e.inputs.candidates)),
            "target_type": action_type(e.target_action),
        }
        for e in train
    ]
)
display(pd.crosstab(menus["menu_types"], menus["target_type"]))

## 5. Optional CPU baselines

Compare uniform probabilities, a smoothed training-only option-type prior, and a learned
shared scorer over `(context, candidate)` pairs. The linear scorer uses hashed features with
explicit context/action interactions and logistic regression. The hash transform is
stateless; model fitting uses only training decisions. Forced one-option decisions are
excluded from fitting because they contain no ranking information.

All models return probabilities over supplied candidates, preserving exact bet sizes.
The primary metric is **exact-candidate accuracy**. Action-type macro-F1 intentionally
ignores sizing and is labeled separately. Log loss and Brier score describe the baseline's
probabilities; they are not solver mixing frequencies or calibrated poker outcomes.

Ties use a deterministic lexical action order. `tie_expected_accuracy` separately shows
expected accuracy if tied maxima were chosen uniformly, giving uniform choice a fair
reference. Report multi-option accuracy and one-option frequency because forced decisions
can inflate the overall score. Review per-street results too.

In [ ]:
if RUN_BASELINE:
    report = run_baselines(ROOT, train, validation, seed=SEED)
    table = pd.DataFrame(
        [
            {"model": model, "subset": subset, **metrics}
            for model, subsets in report["metrics"].items()
            for subset, metrics in subsets.items()
        ]
    )
    display(table.set_index(["model", "subset"]).round(4))
else:
    print("Baseline skipped; validated data and split files are ready.")

## 6. Inspect five mistakes

Five errors are selected reproducibly from the linear scorer (or all errors if fewer).
Inspect Hero's cards, action history, position, sizing, and probabilities. Describe a
plausible cause yourself; the output does not invent explanations.

In [ ]:
if RUN_BASELINE:
    print(f"Showing {len(report['five_mistakes'])} validation mistakes")
    for number, mistake in enumerate(report["five_mistakes"], start=1):
        print(f"\nMistake {number}: {mistake['source_id']}")
        print(mistake["context"])
        print("Candidates:", mistake["candidates"])
        print("Probabilities:", mistake["probabilities"])
        print("Target:", mistake["target_action"], "| Prediction:", mistake["predicted_action"])

## Outputs and next steps

- `data/processed/pokerbench_meeting1/`: raw inspection and failure reports, train/validation
  JSONL, and a manifest with revision, hashes, seed, source counts, and exact membership.
- `artifacts/pokerbench_meeting1/`: validation metrics, model settings, dependency versions,
  and the five mistakes. Generated data and reports stay out of Git.

Run the identical pipeline without Jupyter:
```sh
uv run --extra analysis python -m poker_decision.pipeline
```
Use `--skip-baseline` for just the data pipeline, `--kind preflop` for the smaller source,
or `--sample-per-kind 0` to train on all audited rows.

Next: reuse this input contract for a shared neural candidate scorer, audit truncation,
and evaluate related-scenario grouping. Source menus can contain dataset-specific clues;
a candidate-only baseline helps expose them. We validate menu/label consistency, not
that every supplied menu exhausts the actions legal under poker rules.